# Notebook 02: Feature engineering y particion del dataset

## Descripcion
Preparacion de la ingenieria de variables y la particion del dataset unificado.
**Este notebook no entrena modelos.**

## Flujo
1. Cargar el dataset unificado (`data/df_all_unificado.csv`)
2. Normalizar la potencia: `power_pu = power_kW / nominal_kW`
3. Construir variables derivadas y temporales mediante `build_features()`
4. Particionar en train / validacion / test con criterio temporal y espacial
5. Guardar los splits y la lista de features

## Diseno experimental

| Split | Plantas | Periodo |
|---|---|---|
| Train | Afrisol + E03 | 2022-01-01 a 2023-12-31 |
| Validacion | Afrisol + E03 | 2024-01-01 a 2024-02-29 |
| Test | LECA1 | 2024-03-01 a 2024-05-30 |

El diseno combina dos tipos de generalizacion simultaneamente:
extrapolacion temporal (datos futuros) y extrapolacion espacial
(planta no vista durante el entrenamiento).

## Decisiones de diseno
- `one_hot_encode_plant=False`: el modelo no ve la identidad de la planta,
  lo que fuerza una generalizacion real a LECA1.
- Los lags y rolling generan NaN en las primeras filas de cada planta;
  se eliminan con `drop_na_after_features=True`.
- `power_pu` normaliza la potencia entre plantas con distinta potencia nominal,
  haciendo comparables las series de las tres instalaciones.


### 1. Configuracion de rutas e importaciones


In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

# Localizar la raiz del repositorio
current = Path().resolve()
root = current.anchor
while not ((current / "data").exists() and (current / "src").exists()):
    if str(current) == root:
        raise FileNotFoundError(
            "No se encontro la raiz del proyecto. "
            "Asegurate de que existen las carpetas data/ y src/."
        )
    current = current.parent

os.chdir(current)
if str(current) not in sys.path:
    sys.path.insert(0, str(current))

PROJECT_ROOT = current
DATA_DIR = PROJECT_ROOT / "data"
SPLITS_DIR = DATA_DIR / "splits"

from src.data import add_nominal_power_and_target, split_train_val_test
from src.features import FeatureConfig, build_features, get_feature_columns

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Raiz del proyecto: {PROJECT_ROOT.name}")


### 2. Carga y normalizacion de potencia
Se carga el dataset unificado generado por el notebook 01 y se añade
la columna `power_pu` (potencia en *per-unit* respecto a la potencia nominal
de cada instalacion).


In [ ]:
df_all = pd.read_csv(DATA_DIR / "df_all_unificado.csv", parse_dates=["timestamp"])

# Potencia nominal instalada por planta (kW)
NOMINAL_POWER = {
    "E03":    12.4,
    "Afrisol": 12.0,
    "LECA1":  19.53,
}

# Eliminar columnas de normalizacion previas si ya existieran
df_all = df_all.drop(columns=["power_pu", "p_nominal_kw"], errors="ignore")
df_all = add_nominal_power_and_target(df_all, NOMINAL_POWER)

print(f"Shape: {df_all.shape}")
print(f"Plantas: {sorted(df_all['id_planta'].unique())}")
print(f"Rango temporal: {df_all['timestamp'].min()} -> {df_all['timestamp'].max()}")
df_all[["id_planta", "timestamp", "radiation", "power", "power_pu"]].head()


In [ ]:
# Verificacion: power_pu debe estar acotado entre 0 y ~1
# Valores marginalmente superiores a 1 son posibles por rendimiento pico
df_all["power_pu"].describe()


### 3. Ingenieria de variables
`build_features()` ejecuta el pipeline completo en el orden siguiente:

1. Variables temporales ciclicas (seno/coseno de hora y dia del ano)
2. Bandera de horas de luz (`is_daylight`)
3. Interacciones fisicas (`radiation_x_temp`, `radiation_sq`)
4. Clasificacion de regimen de irradiancia (noche / baja / media / alta)
5. *Lags* de potencia y radiacion agrupados por planta
6. Medias y desviaciones tipicas moviles (con shift previo para evitar fuga)
7. Deltas de potencia y radiacion
8. Eliminacion de NaN generados por *lags* y *rolling*

El parametro `one_hot_encode_plant=False` impide que el modelo use la identidad
de la planta como predictor, forzando la generalizacion real a LECA1.


In [ ]:
config = FeatureConfig(
    lag_steps_power=(1, 2, 4),
    lag_steps_radiation=(1, 2),
    rolling_windows_power=(4, 8),
    rolling_windows_radiation=(4,),
    add_interactions=True,
    add_daylight_flag=True,
    drop_na_after_features=True,
    one_hot_encode_plant=False,
)

df_feat = build_features(df_all, config=config)
df_feat = df_feat.drop(columns=["Mes"], errors="ignore")

print(f"Shape tras feature engineering: {df_feat.shape}")
print(f"\nColumnas generadas ({len(df_feat.columns)}):")
display(df_feat.dtypes.to_frame("dtype"))


### 4. Particion train / validacion / test
La particion combina criterio temporal y espacial:
- **Train y validacion** usan datos de Afrisol y E03 en periodos distintos.
- **Test** usa exclusivamente datos de LECA1, una planta no vista durante
  el entrenamiento, en el periodo posterior al corte de validacion.

Se verifica explicitamente la ausencia de fuga de informacion entre splits.


In [ ]:
train_df, val_df, test_df = split_train_val_test(
    df_feat,
    train_plants=["Afrisol", "E03"],
    test_plant="LECA1",
)

FEATURE_COLS = get_feature_columns(df_feat)
TARGET_COL = "power_pu"

print("Verificacion del diseno experimental:")
print(f"  Train: {train_df.shape} | "
      f"{train_df['timestamp'].min().date()} -> {train_df['timestamp'].max().date()}")
print(f"  Val:   {val_df.shape}   | "
      f"{val_df['timestamp'].min().date()} -> {val_df['timestamp'].max().date()}")
print(f"  Test:  {test_df.shape}  | "
      f"{test_df['timestamp'].min().date()} -> {test_df['timestamp'].max().date()}")
print(f"\n  Plantas en train: {sorted(train_df['id_planta'].unique())}")
print(f"  Plantas en val:   {sorted(val_df['id_planta'].unique())}")
print(f"  Plantas en test:  {sorted(test_df['id_planta'].unique())}")

# Verificacion de ausencia de fuga de informacion
assert train_df["timestamp"].max() < val_df["timestamp"].min(), \
    "Fuga de informacion detectada entre train y validacion."
assert val_df["timestamp"].max() < test_df["timestamp"].min(), \
    "Fuga de informacion detectada entre validacion y test."
print("\nAusencia de data leakage confirmada.")


### 5. Distribucion de ceros (produccion nocturna)
Un porcentaje elevado de ceros es esperado en datos de 15 minutos: los
periodos nocturnos representan aproximadamente la mitad de los registros.
Esta proporcion es relevante porque los modelos tienden a predecir bien
los ceros nocturnos, lo que puede inflar artificialmente el R^2 global.
Por ello, las metricas finales se calculan tambien restringidas a horas de luz.


In [ ]:
for name, df in [("Train", train_df), ("Validacion", val_df), ("Test", test_df)]:
    zeros_pct = (df[TARGET_COL] == 0).mean() * 100
    print(f"{name:12s}: {len(df):>8,} filas | {zeros_pct:.1f}% ceros (noche)")


### 6. Guardar splits y lista de features
Se exportan los tres conjuntos a `data/splits/` junto con la lista de
columnas de entrada en formato JSON para garantizar la reproducibilidad
de los experimentos en notebooks posteriores.


In [ ]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)

with open(SPLITS_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_COLS, f, indent=2, ensure_ascii=False)

print(f"Splits guardados en: data/splits/")
print(f"  train.csv   : {len(train_df):,} filas")
print(f"  val.csv     : {len(val_df):,} filas")
print(f"  test.csv    : {len(test_df):,} filas")
print(f"  Features    : {len(FEATURE_COLS)} columnas")


## Conclusiones del notebook 02

| Variable / Split | Descripcion |
|---|---|
| `power_pu` | Potencia normalizada por potencia nominal. Permite comparar plantas. |
| Lags generados | `power_pu_lag_1/2/4`, `radiation_lag_1/2` |
| Rolling | Media y desviacion tipica en ventanas de 4 y 8 pasos (1 h y 2 h) |
| Features totales | ~32 variables de entrada |
| Train | Afrisol + E03, 2022-2023 |
| Validacion | Afrisol + E03, ene-feb 2024 |
| Test | LECA1, mar-may 2024 (planta no vista en entrenamiento) |

El diseno experimental somete a los modelos a una doble generalizacion:
temporal (datos futuros) y espacial (instalacion no vista), lo que constituye
un escenario de evaluacion riguroso para un sistema de prediccion real.

**Archivos de salida:** `data/splits/train.csv`, `val.csv`, `test.csv`, `feature_cols.json`  
**Punto de entrada del siguiente notebook:** entrenamiento y comparativa de modelos.
